In [2]:
import psycopg2
from psycopg2 import Error

def create_db_connection(host_name, port_name, user_name, user_password, dbname):
    connection = None
    try:
        connection = psycopg2.connect(dbname=dbname, user=user_name, password=user_password, host=host_name, port=port_name)
        print("openGauss Database connection successful")
    except Error as err:
        print(f"Error: {err}")
    return connection

connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

openGauss Database connection successful


In [4]:
def read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as err:
        print(f"Error: '{err}'")

In [ ]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as err:
        print(f"Error: {err}")

In [28]:
cursor = connection.cursor()
with open('employ.sql', 'r') as file:
    sql_script = file.read()

sql_script = "DROP SCHEMA IF EXISTS kk CASCADE;\n" + sql_script

sql_commands = sql_script.split(';')
for command in sql_commands:
    if command.strip():
        cursor.execute(command)

print("employ.sql 执行成功！")
connection.commit()
connection.close()

employ.sql 执行成功！


In [22]:
#(1)查询每个部门(departments)的编号(dept_no),名称(dept_name),在该部门工作过的雇员(employees)人数,最低工资(salary),平均工资,最高工资及工资总额；（需要建立一个中间表）
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    d.dept_no, 
    d.dept_name, 
    COUNT(DISTINCT e.emp_no) AS employee_count,
    MIN(s.salary) AS min_salary,
    AVG(s.salary) AS avg_salary,
    MAX(s.salary) AS max_salary,
    SUM(s.salary) AS total_salary
FROM 
    kk.departments d
JOIN 
    kk.dept_emp de ON d.dept_no = de.dept_no
JOIN 
    kk.employees e ON de.emp_no = e.emp_no
JOIN 
    kk.salaries s ON e.emp_no = s.emp_no
GROUP BY 
    d.dept_no, d.dept_name;
"""

results = read_query(connection, query)
if results:
    print("Department Report:")
    print("Dept No | Dept Name | Employee Count | Min Salary | Avg Salary | Max Salary | Total Salary")
    print("-" * 80)
    for result in results:
        print(result)
else:
    print("No results found.")

openGauss Database connection successful
Department Report:
Dept No | Dept Name | Employee Count | Min Salary | Avg Salary | Max Salary | Total Salary
--------------------------------------------------------------------------------
('d001', 'Marketing', 20211, 39127, Decimal('71913.200004191532'), 145128, 13725425266)
('d002', 'Finance', 17346, 38812, Decimal('70489.364896996098'), 142395, 11650834677)
('d003', 'Human Resources', 17786, 38735, Decimal('55574.879369695531'), 141953, 9363811425)
('d004', 'Production', 73485, 38623, Decimal('59605.482461651448'), 138273, 41554438942)
('d005', 'Development', 85707, 38849, Decimal('59478.901162431823'), 144434, 48179456393)
('d006', 'Quality Management', 20117, 38786, Decimal('57251.271913415990'), 132103, 10865203635)
('d008', 'Research', 21126, 38851, Decimal('59665.181701268599'), 130211, 11969730427)
('d009', 'Customer Service', 23580, 38836, Decimal('58770.366479762480'), 144866, 13143639841)


In [23]:
#(2)查询每个部门(departments)的编号(dept_no),名称(dept_name),及各个时间段(from_date,to_date)担任该部门经理(dept_manager)的雇员的编号(emp_no)和姓名(first_name+last_name)，并按时间段先后显示；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    d.dept_no, 
    d.dept_name, 
    dm.from_date,
    dm.to_date,
    e.emp_no,
    CONCAT(e.first_name, ' ', e.last_name) AS manager_name
FROM 
    kk.departments d
JOIN 
    kk.dept_manager dm ON d.dept_no = dm.dept_no
JOIN 
    kk.employees e ON dm.emp_no = e.emp_no
ORDER BY 
    d.dept_no, dm.from_date;
"""

results = read_query(connection, query)
if results:
    print("Department Manager Report:")
    print("Dept No | Dept Name | From Date | To Date | Emp No | Manager Name")
    print("-" * 80)
    for result in results:
        print(result)
else:
    print("No results found.")

openGauss Database connection successful
Department Manager Report:
Dept No | Dept Name | From Date | To Date | Emp No | Manager Name
--------------------------------------------------------------------------------
('d001', 'Marketing', datetime.datetime(1985, 1, 1, 0, 0), datetime.datetime(1991, 10, 1, 0, 0), 110022, 'Margareta Markovitch')
('d001', 'Marketing', datetime.datetime(1991, 10, 1, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 110039, 'Vishwani Minakawa')
('d002', 'Finance', datetime.datetime(1985, 1, 1, 0, 0), datetime.datetime(1989, 12, 17, 0, 0), 110085, 'Ebru Alpin')
('d002', 'Finance', datetime.datetime(1989, 12, 17, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 110114, 'Isamu Legleitner')
('d003', 'Human Resources', datetime.datetime(1985, 1, 1, 0, 0), datetime.datetime(1992, 3, 21, 0, 0), 110183, 'Shirish Ossenbruggen')
('d003', 'Human Resources', datetime.datetime(1992, 3, 21, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 110228, 'Karsten Sigstam')
('d004', 'Production', 

In [25]:
#(3)查询每位雇员的编号(emp_no),姓名(first_name+last_name),及各个时间段(from_date,end_data)的工资额(salary),并按时间段先后显示；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    e.emp_no,
    CONCAT(e.first_name, ' ', e.last_name) AS full_name,
    s.from_date,
    s.to_date,
    s.salary
FROM 
    kk.employees e
JOIN 
    kk.salaries s ON e.emp_no = s.emp_no
ORDER BY 
    e.emp_no, s.from_date;
"""

results = read_query(connection, query)
if results:
    print("Employee Salary Report:")
    print("Emp No | Full Name | From Date | To Date | Salary")
    print("-" * 60)
    count = 0
    for result in results:
        if count < 100:
            print(result)
            count += 1
        else:
            break
else:
    print("No results found.")

openGauss Database connection successful
Employee Salary Report:
Emp No | Full Name | From Date | To Date | Salary
------------------------------------------------------------
(10001, 'Georgi Facello', datetime.datetime(1986, 6, 26, 0, 0), datetime.datetime(1987, 6, 26, 0, 0), 60117)
(10001, 'Georgi Facello', datetime.datetime(1987, 6, 26, 0, 0), datetime.datetime(1988, 6, 25, 0, 0), 62102)
(10001, 'Georgi Facello', datetime.datetime(1988, 6, 25, 0, 0), datetime.datetime(1989, 6, 25, 0, 0), 66074)
(10001, 'Georgi Facello', datetime.datetime(1989, 6, 25, 0, 0), datetime.datetime(1990, 6, 25, 0, 0), 66596)
(10001, 'Georgi Facello', datetime.datetime(1990, 6, 25, 0, 0), datetime.datetime(1991, 6, 25, 0, 0), 66961)
(10001, 'Georgi Facello', datetime.datetime(1991, 6, 25, 0, 0), datetime.datetime(1992, 6, 24, 0, 0), 71046)
(10001, 'Georgi Facello', datetime.datetime(1992, 6, 24, 0, 0), datetime.datetime(1993, 6, 24, 0, 0), 74333)
(10001, 'Georgi Facello', datetime.datetime(1993, 6, 24, 0, 0

In [34]:
#(4)查询每位雇员的编号(emp_no),姓名(first_name+last_name),及各个时间段(from_date,end_data)担任的职务(title),并按时间段先后显示；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    e.emp_no, 
    e.first_name || ' ' || e.last_name AS employee_name, 
    t.from_date, 
    t.to_date, 
    t.title
FROM 
    kk.employees e
JOIN 
    kk.titles t ON e.emp_no = t.emp_no
ORDER BY 
    e.emp_no, t.from_date;
"""

results = read_query(connection, query)
if results:
    print("Employee Title Report:")
    print("Emp No | Full Name | From Date | To Date | Title")
    print("-" * 60)
    count = 0
    for result in results:
        if count < 100:
            print(result)
            count += 1
        else:
            break
else:
    print("No results found.")

openGauss Database connection successful
Employee Title Report:
Emp No | Full Name | From Date | To Date | Title
------------------------------------------------------------
(10001, 'Georgi Facello', datetime.datetime(1986, 6, 26, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 'Senior Engineer')
(10002, 'Bezalel Simmel', datetime.datetime(1996, 8, 3, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 'Staff')
(10003, 'Parto Bamford', datetime.datetime(1995, 12, 3, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 'Senior Engineer')
(10004, 'Chirstian Koblick', datetime.datetime(1986, 12, 1, 0, 0), datetime.datetime(1995, 12, 1, 0, 0), 'Engineer')
(10004, 'Chirstian Koblick', datetime.datetime(1995, 12, 1, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 'Senior Engineer')
(10005, 'Kyoichi Maliniak', datetime.datetime(1989, 9, 12, 0, 0), datetime.datetime(1996, 9, 12, 0, 0), 'Staff')
(10005, 'Kyoichi Maliniak', datetime.datetime(1996, 9, 12, 0, 0), datetime.datetime(9999, 1, 1, 0, 0), 'Senior Staff')
(10006

In [35]:
#(5)查询担任每种职务(title)的雇员人数；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    t.title, 
    COUNT(DISTINCT t.emp_no) AS employee_count
FROM 
    kk.titles t
GROUP BY 
    t.title;
"""

results = read_query(connection, query)
if results:
    print("Title Report:")
    print("Title           | Employee Count")
    print("-" * 30)
    for result in results:
        title, employee_count = result
        print(f"{title.ljust(15)} | {employee_count}")
else:
    print("No results found.")

openGauss Database connection successful
Title Report:
Title           | Employee Count
------------------------------
Assistant Engineer | 15128
Engineer        | 115003
Manager         | 24
Senior Engineer | 97750
Senior Staff    | 92853
Staff           | 107391
Technique Leader | 15157


In [36]:
#(6)查询每个部门中担任每种职务(title)的雇员人数；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    d.dept_no,
    d.dept_name,
    t.title,
    COUNT(DISTINCT e.emp_no) AS employee_count
FROM 
    kk.departments d
JOIN 
    kk.dept_emp de ON d.dept_no = de.dept_no
JOIN 
    kk.employees e ON de.emp_no = e.emp_no
JOIN 
    kk.titles t ON e.emp_no = t.emp_no
GROUP BY 
    d.dept_no, d.dept_name, t.title
ORDER BY 
    d.dept_no, t.title;
"""

results = read_query(connection, query)
if results:
    print("Department Title Report:")
    print("Dept No | Dept Name     | Title           | Employee Count")
    print("-" * 60)
    for result in results:
        dept_no, dept_name, title, employee_count = result
        print(f"{dept_no} | {dept_name.ljust(12)} | {title.ljust(15)} | {employee_count}")
else:
    print("No results found.")

openGauss Database connection successful
Department Title Report:
Dept No | Dept Name     | Title           | Employee Count
------------------------------------------------------------
d001 | Marketing    | Manager         | 2
d001 | Marketing    | Senior Staff    | 13940
d001 | Marketing    | Staff           | 16196
d002 | Finance      | Manager         | 2
d002 | Finance      | Senior Staff    | 12139
d002 | Finance      | Staff           | 13929
d003 | Human Resources | Manager         | 2
d003 | Human Resources | Senior Staff    | 12274
d003 | Human Resources | Staff           | 14342
d004 | Production   | Assistant Engineer | 6445
d004 | Production   | Engineer        | 49649
d004 | Production   | Manager         | 4
d004 | Production   | Senior Engineer | 42205
d004 | Production   | Senior Staff    | 1270
d004 | Production   | Staff           | 1478
d004 | Production   | Technique Leader | 6556
d005 | Development  | Assistant Engineer | 7769
d005 | Development  | Engineer       

In [6]:
#(7)查询所有曾经在‘Development’工作过雇员的编号(emp_no),姓名(first_name+last_name),及时间段(from_date,end_data)；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    e.emp_no, 
    e.first_name || ' ' || e.last_name AS employee_name, 
    de.from_date, 
    de.to_date
FROM 
    kk.employees e
JOIN 
    kk.dept_emp de ON e.emp_no = de.emp_no
JOIN 
    kk.departments d ON de.dept_no = d.dept_no
WHERE 
    d.dept_name = 'Development'
ORDER BY 
    e.emp_no, de.from_date;
"""

results = read_query(connection, query)
if results:
    print("Development Department Employees Report:")
    print("Emp No | Employee Name     | From Date  | To Date    ")
    print("-" * 60)
    count = 0
    for result in results:
        emp_no, employee_name, from_date, to_date = result
        if count < 100:
            print(f"{emp_no} | {employee_name.ljust(18)} | {from_date} | {to_date}")
            count += 1
        else:
            break
else:
    print("No results found.")


openGauss Database connection successful
Development Department Employees Report:
Emp No | Employee Name     | From Date  | To Date    
------------------------------------------------------------
10001 | Georgi Facello     | 1986-06-26 00:00:00 | 9999-01-01 00:00:00
10006 | Anneke Preusig     | 1990-08-05 00:00:00 | 9999-01-01 00:00:00
10008 | Saniya Kalloufi    | 1998-03-11 00:00:00 | 2000-07-31 00:00:00
10012 | Patricio Bridgland | 1992-12-18 00:00:00 | 9999-01-01 00:00:00
10014 | Berni Genin        | 1993-12-29 00:00:00 | 9999-01-01 00:00:00
10018 | Kazuhide Peha      | 1987-04-03 00:00:00 | 1992-07-29 00:00:00
10021 | Ramzi Erde         | 1988-02-10 00:00:00 | 2002-07-15 00:00:00
10022 | Shahaf Famili      | 1999-09-03 00:00:00 | 9999-01-01 00:00:00
10023 | Bojan Montemayor   | 1999-09-27 00:00:00 | 9999-01-01 00:00:00
10025 | Prasadram Heyers   | 1987-08-17 00:00:00 | 1997-10-15 00:00:00
10027 | Divier Reistad     | 1995-04-02 00:00:00 | 9999-01-01 00:00:00
10028 | Domenick Tempe

In [38]:
#(8)查询曾经在所有部门都工作过的雇员的编号(emp_no),姓名(first_name+last_name)；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
WITH DepartmentCount AS (
    SELECT COUNT(*) AS total_departments
    FROM kk.departments
),
EmployeeDepartmentCount AS (
    SELECT 
        e.emp_no, 
        e.first_name || ' ' || e.last_name AS employee_name,
        COUNT(DISTINCT de.dept_no) AS worked_departments
    FROM 
        kk.employees e
    JOIN 
        kk.dept_emp de ON e.emp_no = de.emp_no
    GROUP BY 
        e.emp_no, e.first_name, e.last_name
)
SELECT 
    edc.emp_no, 
    edc.employee_name
FROM 
    EmployeeDepartmentCount edc,
    DepartmentCount dc
WHERE 
    edc.worked_departments = dc.total_departments;
"""

results = read_query(connection, query)
if results:
    print("Employees Who Worked in All Departments:")
    print("Emp No | Employee Name")
    print("-" * 30)
    for result in results:
        emp_no, employee_name = result
        print(f"{emp_no} | {employee_name}")
else:
    print("No employees found who worked in all departments.")

openGauss Database connection successful
No employees found who worked in all departments.


In [88]:
#(9)在dept_emp表中插入适当数据使得至少3个以上雇员满足上一题的查询要求；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

employee_insert_query = """
INSERT INTO kk.employees (emp_no, birth_date, first_name, last_name, gender, hire_date)
SELECT * FROM (
    VALUES 
    (500001, '1985-03-15', 'Shengxiang1', 'Lin', 'M', '2010-06-01'),
    (500002, '1988-07-22', 'Shengxiang2', 'Lin', 'M', '2012-09-15'),
    (500003, '1990-11-30', 'Shengxiang3', 'Lin', 'M', '2015-03-10')
) AS new_employees (emp_no, birth_date, first_name, last_name, gender, hire_date)
WHERE NOT EXISTS (
    SELECT 1 FROM kk.employees WHERE emp_no = new_employees.emp_no
);
"""

dept_emp_insert_query = """
INSERT INTO kk.dept_emp (emp_no, dept_no, from_date, to_date)
SELECT * FROM (
    VALUES 
    (500001, 'd001', '2011-01-01', '2012-12-31'),
    (500001, 'd002', '2013-01-01', '2014-06-30'),
    (500001, 'd003', '2014-07-01', '2015-12-31'),
    (500001, 'd004', '2016-01-01', '2017-06-30'),
    (500001, 'd005', '2017-07-01', '2018-12-31'),
    (500001, 'd006', '2019-01-01', '2020-06-30'),
    (500001, 'd008', '2020-07-01', '2021-12-31'),
    (500001, 'd009', '2022-01-01', '2023-12-31'),

    (500002, 'd001', '2013-04-01', '2014-03-31'),
    (500002, 'd002', '2014-04-01', '2015-03-31'),
    (500002, 'd003', '2015-04-01', '2016-03-31'),
    (500002, 'd004', '2016-04-01', '2017-03-31'),
    (500002, 'd005', '2017-04-01', '2018-03-31'),
    (500002, 'd006', '2018-04-01', '2019-03-31'),
    (500002, 'd008', '2019-04-01', '2020-03-31'),
    (500002, 'd009', '2020-04-01', '2021-03-31'),

    (500003, 'd001', '2016-07-01', '2017-06-30'),
    (500003, 'd002', '2017-07-01', '2018-06-30'),
    (500003, 'd003', '2018-07-01', '2019-06-30'),
    (500003, 'd004', '2019-07-01', '2020-06-30'),
    (500003, 'd005', '2020-07-01', '2021-06-30'),
    (500003, 'd006', '2021-07-01', '2022-06-30'),
    (500003, 'd008', '2022-07-01', '2023-06-30'),
    (500003, 'd009', '2023-07-01', '2024-06-30')
) AS new_dept_emp (emp_no, dept_no, from_date, to_date)
WHERE NOT EXISTS (
    SELECT 1 FROM kk.dept_emp WHERE emp_no = new_dept_emp.emp_no AND dept_no = new_dept_emp.dept_no
);
"""

execute_query(connection, employee_insert_query)
execute_query(connection, dept_emp_insert_query)
connection.commit()

openGauss Database connection successful
Query executed successfully
Query executed successfully


In [87]:
#(9)在dept_emp表中插入适当数据使得至少3个以上雇员满足上一题的查询要求；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
SELECT 
    e.emp_no, 
    e.first_name || ' ' || e.last_name AS employee_name,
    d.dept_name,
    de.from_date,
    de.to_date
FROM 
    kk.employees e
JOIN 
    kk.dept_emp de ON e.emp_no = de.emp_no
JOIN 
    kk.departments d ON de.dept_no = d.dept_no
WHERE 
    (e.last_name = 'Lin')
ORDER BY 
    e.emp_no, de.from_date;

"""
results = read_query(connection, query) 
connection.commit()
if results:
    print("Employees Information:")
    print("Emp No | Employee Name       | Department           | From Date   | To Date")
    print("-" * 80)
    current_emp = None
    for result in results:
        emp_no, employee_name, dept_name, from_date, to_date = result
        if emp_no != current_emp:
            print("-" * 80)  # 分隔不同员工的信息
            current_emp = emp_no
        print(f"{emp_no} | {employee_name.ljust(20)} | {dept_name.ljust(20)} | {from_date} | {to_date}")
else:
    print("No employees found with the names Shengxiang1/2/3 Lin.") 

openGauss Database connection successful
Employees Information:
Emp No | Employee Name       | Department           | From Date   | To Date
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
500001 | Shengxiang1 Lin      | Marketing            | 2011-01-01 00:00:00 | 2012-12-31 00:00:00
500001 | Shengxiang1 Lin      | Finance              | 2013-01-01 00:00:00 | 2014-06-30 00:00:00
500001 | Shengxiang1 Lin      | Human Resources      | 2014-07-01 00:00:00 | 2015-12-31 00:00:00
500001 | Shengxiang1 Lin      | Production           | 2016-01-01 00:00:00 | 2017-06-30 00:00:00
500001 | Shengxiang1 Lin      | Development          | 2017-07-01 00:00:00 | 2018-12-31 00:00:00
500001 | Shengxiang1 Lin      | Quality Management   | 2019-01-01 00:00:00 | 2020-06-30 00:00:00
500001 | Shengxiang1 Lin      | Research             | 2020-07-01 00:00:00 | 2021-12-31 00:00:00
500001 | Shengxiang

In [78]:
#(9)在dept_emp表中插入适当数据使得至少3个以上雇员满足上一题的查询要求；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

query = """
WITH DepartmentCount AS (
    SELECT COUNT(*) AS total_departments
    FROM kk.departments
),
EmployeeDepartmentCount AS (
    SELECT 
        e.emp_no, 
        e.first_name || ' ' || e.last_name AS employee_name,
        COUNT(DISTINCT de.dept_no) AS worked_departments
    FROM 
        kk.employees e
    JOIN 
        kk.dept_emp de ON e.emp_no = de.emp_no
    GROUP BY 
        e.emp_no, e.first_name, e.last_name
)
SELECT 
    edc.emp_no, 
    edc.employee_name
FROM 
    EmployeeDepartmentCount edc,
    DepartmentCount dc
WHERE 
    edc.worked_departments = dc.total_departments;
"""

results = read_query(connection, query)
if results:
    print("Employees Who Worked in All Departments:")
    print("Emp No | Employee Name")
    print("-" * 30)
    for result in results:
        emp_no, employee_name = result
        print(f"{emp_no} | {employee_name}")
else:
    print("No employees found who worked in all departments.")

openGauss Database connection successful
Employees Who Worked in All Departments:
Emp No | Employee Name
------------------------------
500001 | Shengxiang1 Lin
500002 | Shengxiang2 Lin
500003 | Shengxiang3 Lin


In [ ]:
#(10)添加一个部门(departments)“学习部”(Learning)；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

insert = """
INSERT INTO kk.departments VALUES ('d000','Learning');
"""

execute_query(connection, insert.encode(encoding='utf-8'))

q1 = """
SELECT *
FROM kk.departments;
"""
results = read_query(connection, q1.encode(encoding='utf-8'))
for result in results:
    print(result)

In [ ]:
#(11)删除“销售部”(Sales)；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

dels = """
DELETE FROM kk.departments
WHERE dept_name = 'Sales';
"""
execute_query(connection, dels.encode(encoding='utf-8'))

ql = """
SELECT *
FROM kk.departments;
"""

results = read_query(connection, ql.encode(encoding='utf-8'))
for result in results:
    print(result)

In [75]:
#(12)删除工资(salary)低于7w的员工；
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

delete_query = """
DELETE FROM kk.dept_emp
WHERE emp_no IN (
    SELECT emp_no FROM kk.salaries
    WHERE salary < 70000
);

DELETE FROM kk.salaries
WHERE salary < 70000;

DELETE FROM kk.employees
WHERE emp_no IN (
    SELECT emp_no FROM kk.salaries
    WHERE salary < 70000
);
"""

preview_query = """
SELECT e.emp_no, e.first_name, e.last_name, s.salary
FROM kk.employees e
JOIN kk.salaries s ON e.emp_no = s.emp_no
WHERE s.salary < 70000
ORDER BY s.salary;
"""

try:
    print("即将删除的员工列表(薪资<7万):")
    to_delete = read_query(connection, preview_query)
    count=0
    if to_delete:
        print("Emp No | First Name | Last Name | Salary")
        print("-" * 50)
        for emp in to_delete:
            count = count + 1
            if cout>=100:
                break
            print(f"{emp[0]} | {emp[1]} | {emp[2]} | {emp[3]}")
        execute_query(connection, delete_query)
        connection.commit()
        print(f"\n已成功删除 {len(to_delete)} 名薪资低于7万的员工")
    else:
        print("没有找到薪资低于7万的员工")
        
except Exception as e:
    connection.rollback()
    print(f"删除操作失败: {e}")
finally:
    connection.close()

openGauss Database connection successful
即将删除的员工列表(薪资<7万):
没有找到薪资低于7万的员工


In [ ]:
#(13)修改“学习部”(Learning)的编号为“d01”。
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

update = """
UPDATE kk.departments
SET dept_no = 'd01'
WHERE dept_name = 'Learning';
"""

execute_query(connection, update.encode(encoding='utf-8'))

q1 = """
SELECT *
FROM kk.departments;
"""
results = read_query(connection, q1.encode(encoding='utf-8'))
for result in results:
    print(result)

In [9]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as err:
        print(f"Error: {err}")

In [11]:
#创建模式“sc”，并在“sc”基础上创建“students”表：
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

create_schema = """
CREATE SCHEMA sc1;  -- 1. Create schema "sc1"
"""

create_table = """
CREATE TABLE sc1.students (
    id INT PRIMARY KEY,  -- 2. Create "students" table
    username VARCHAR(50) NOT NULL,
    email VARCHAR(100) NOT NULL
);
"""

execute_query(connection, create_schema)
execute_query(connection, create_table)

openGauss Database connection successful
Query executed successfully
Query executed successfully


In [12]:
def read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as err:
        print(f"Error: '{err}'")

In [7]:
#查询所有部门信息：
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

q1 = """
SELECT *
FROM kk.departments;
"""
results = read_query(connection, q1)
for result in results:
    print(result)

('d001', 'Marketing')
('d002', 'Finance')
('d003', 'Human Resources')
('d004', 'Production')
('d005', 'Development')
('d006', 'Quality Management')
('d008', 'Research')
('d009', 'Customer Service')
('d01 ', 'Learning')


In [17]:
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

dels = """
DELETE FROM kk.departments
WHERE dept_name = 'Learning';
"""
execute_query(connection, dels.encode(encoding='utf-8'))

ql = """
SELECT *
FROM kk.departments;
"""

results = read_query(connection, ql.encode(encoding='utf-8'))
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
('d001', 'Marketing')
('d002', 'Finance')
('d003', 'Human Resources')
('d004', 'Production')
('d005', 'Development')
('d006', 'Quality Management')
('d008', 'Research')
('d009', 'Customer Service')


In [90]:
#加入学习部“Learning”：
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

insert = """
INSERT INTO kk.departments VALUES ('d000','Learning');
"""

execute_query(connection, insert.encode(encoding='utf-8'))

q1 = """
SELECT *
FROM kk.departments;
"""
results = read_query(connection, q1.encode(encoding='utf-8'))
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
('d001', 'Marketing')
('d002', 'Finance')
('d003', 'Human Resources')
('d004', 'Production')
('d005', 'Development')
('d006', 'Quality Management')
('d008', 'Research')
('d009', 'Customer Service')
('d000', 'Learning')


In [91]:
#修改学习部的编号为d01：
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

update = """
UPDATE kk.departments
SET dept_no = 'd01'
WHERE dept_name = 'Learning';
"""

execute_query(connection, update.encode(encoding='utf-8'))

q1 = """
SELECT *
FROM kk.departments;
"""
results = read_query(connection, q1.encode(encoding='utf-8'))
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
('d001', 'Marketing')
('d002', 'Finance')
('d003', 'Human Resources')
('d004', 'Production')
('d005', 'Development')
('d006', 'Quality Management')
('d008', 'Research')
('d009', 'Customer Service')
('d01 ', 'Learning')


In [92]:
#删除销售部“Sales”：
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

dels = """
DELETE FROM kk.departments
WHERE dept_name = 'Sales';
"""
execute_query(connection, dels.encode(encoding='utf-8'))

ql = """
SELECT *
FROM kk.departments;
"""

results = read_query(connection, ql.encode(encoding='utf-8'))
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
('d001', 'Marketing')
('d002', 'Finance')
('d003', 'Human Resources')
('d004', 'Production')
('d005', 'Development')
('d006', 'Quality Management')
('d008', 'Research')
('d009', 'Customer Service')
('d01 ', 'Learning')


In [10]:
connection = create_db_connection("10.180.59.104", "5432", "user3", "123456abc.", "db3")

dels = """
DELETE FROM kk.departments
WHERE dept_name = 'Learning';
"""
execute_query(connection, dels.encode(encoding='utf-8'))

ql = """
SELECT *
FROM kk.departments;
"""

results = read_query(connection, ql.encode(encoding='utf-8'))
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
('d001', 'Marketing')
('d002', 'Finance')
('d003', 'Human Resources')
('d004', 'Production')
('d005', 'Development')
('d006', 'Quality Management')
('d008', 'Research')
('d009', 'Customer Service')
